<a href="https://colab.research.google.com/github/ehsan-ai-sudo/mamo-ai-project/blob/main/mamo_ai_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# 🧠 استخدام نموذج Kaggle المحترف لكشف الأورام الدماغية
# =====================================================

# الخطوة 1: تثبيت المكتبات
print("📦 تثبيت المكتبات...")
!pip install -q kaggle tensorflow opencv-python-headless pillow matplotlib seaborn

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import json
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("✅ تم تثبيت المكتبات بنجاح")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU متاح: {len(tf.config.list_physical_devices('GPU')) > 0}")

# =====================================================
# الخطوة 2: إعداد Kaggle API
# =====================================================

print("\n" + "="*60)
print("📤 الآن قم برفع ملف kaggle.json")
print("="*60)
print("\n📝 كيفية الحصول على kaggle.json:")
print("1. اذهب إلى: https://www.kaggle.com/settings")
print("2. انزل للأسفل حتى قسم 'API'")
print("3. اضغط 'Create New API Token'")
print("4. سيتم تحميل ملف kaggle.json تلقائياً")
print("\n👇 ارفع الملف الآن:")

# رفع kaggle.json
uploaded = files.upload()

# إعداد Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ تم إعداد Kaggle API بنجاح!")

# =====================================================
# الخطوة 3: تحميل النموذج المدرب من Kaggle
# =====================================================

print("\n" + "="*60)
print("📥 تحميل النموذج المدرب من Kaggle...")
print("="*60)

# الطريقة 1: تحميل نموذج من Kaggle Models
def download_kaggle_model():
    """
    تحميل نموذج مدرب من Kaggle Models
    """
    # نموذج Brain Tumor Classification (مثال)
    model_owner = "masoudnickparvar"
    model_name = "brain-tumor-mri-dataset"

    print(f"\n📦 تحميل النموذج: {model_owner}/{model_name}")

    # تحميل Dataset الذي يحتوي على النموذج
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset

    # فك الضغط
    !unzip -q brain-tumor-mri-dataset.zip -d brain_tumor_data

    print("✅ تم التحميل بنجاح!")
    return 'brain_tumor_data'

# الطريقة 2: تحميل نموذج محدد (إذا كان متاحاً)
def download_specific_model():
    """
    تحميل نموذج .h5 محدد إذا كان موجود
    """
    try:
        # استبدل هذا بالرابط الصحيح للنموذج
        !kaggle models instances versions download username/model-name/framework/variation
        return True
    except:
        print("⚠️ النموذج غير متاح، سنقوم ببناء نموذج")
        return False

# تحميل البيانات
data_path = download_kaggle_model()

# =====================================================
# الخطوة 4: بناء وتدريب نموذج احترافي
# =====================================================

print("\n" + "="*60)
print("🔧 بناء نموذج احترافي...")
print("="*60)

# معلمات النموذج
IMG_SIZE = 224
BATCH_SIZE = 32
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASSES)

# إعداد البيانات
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# تحميل البيانات
train_dir = os.path.join(data_path, 'Training')

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print(f"\n✅ عدد صور التدريب: {train_generator.samples}")
print(f"✅ عدد صور التحقق: {validation_generator.samples}")

# =====================================================
# الخطوة 5: بناء نموذج EfficientNet متقدم
# =====================================================

def create_advanced_model():
    """
    إنشاء نموذج متقدم باستخدام EfficientNetB3
    """
    base_model = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # تجميد الطبقات الأساسية
    base_model.trainable = False

    # بناء النموذج الكامل
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    return model

print("🏗️ إنشاء النموذج...")
model = create_advanced_model()

# ترجمة النموذج
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

model.summary()

# =====================================================
# الخطوة 6: Callbacks متقدمة
# =====================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    TensorBoard
)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_brain_tumor_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    TensorBoard(log_dir='./logs')
]

# =====================================================
# الخطوة 7: تدريب النموذج
# =====================================================

print("\n" + "="*60)
print("🚀 بدء التدريب...")
print("="*60)

EPOCHS = 30

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ اكتمل التدريب!")

# =====================================================
# الخطوة 8: تقييم النموذج
# =====================================================

print("\n" + "="*60)
print("📊 تقييم النموذج...")
print("="*60)

# تحميل أفضل نموذج
best_model = tf.keras.models.load_model('best_brain_tumor_model.h5')

# التقييم
test_dir = os.path.join(data_path, 'Testing')
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

results = best_model.evaluate(test_generator)
print("\n📈 نتائج الاختبار:")
print(f"Loss: {results[0]:.4f}")
print(f"Accuracy: {results[1]*100:.2f}%")
print(f"Precision: {results[2]*100:.2f}%")
print(f"Recall: {results[3]*100:.2f}%")
print(f"AUC: {results[4]:.4f}")

# =====================================================
# الخطوة 9: رسم النتائج
# =====================================================

def plot_training_results(history):
    """
    رسم منحنيات التدريب
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Precision
    axes[1, 0].plot(history.history['precision'], label='Train', linewidth=2)
    axes[1, 0].plot(history.history['val_precision'], label='Validation', linewidth=2)
    axes[1, 0].set_title('Model Precision', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Recall
    axes[1, 1].plot(history.history['recall'], label='Train', linewidth=2)
    axes[1, 1].plot(history.history['val_recall'], label='Validation', linewidth=2)
    axes[1, 1].set_title('Model Recall', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_results.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_results(history)

# =====================================================
# الخطوة 10: Confusion Matrix
# =====================================================

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# التنبؤات
predictions = best_model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(true_classes, predicted_classes,
                           target_names=CLASSES))

# =====================================================
# الخطوة 11: دالة التنبؤ على صورة واحدة
# =====================================================

def predict_brain_tumor(image_path, model, show_visualization=True):
    """
    التنبؤ بوجود ورم في صورة MRI
    """
    # قراءة الصورة
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # تحضير الصورة
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_array = np.expand_dims(img_resized / 255.0, axis=0)

    # التنبؤ
    predictions = model.predict(img_array, verbose=0)
    predicted_class_idx = np.argmax(predictions[0])
    predicted_class = CLASSES[predicted_class_idx]
    confidence = predictions[0][predicted_class_idx] * 100

    # النتيجة
    result = {
        'predicted_class': predicted_class,
        'confidence': confidence,
        'all_probabilities': {CLASSES[i]: float(predictions[0][i] * 100)
                             for i in range(len(CLASSES))},
        'has_tumor': predicted_class != 'notumor',
        'severity': 'High' if confidence > 90 else 'Medium' if confidence > 70 else 'Low'
    }

    # عرض النتائج
    if show_visualization:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # الصورة الأصلية
        axes[0].imshow(img_rgb)
        axes[0].set_title('Original MRI Scan', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # التنبؤ
        color = 'red' if result['has_tumor'] else 'green'
        axes[1].imshow(img_rgb)
        axes[1].set_title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%",
                         fontsize=14, fontweight='bold', color=color)
        axes[1].axis('off')

        # رسم بياني للاحتماليات
        colors = ['#ff6b6b' if i == predicted_class_idx else '#4ecdc4'
                 for i in range(len(CLASSES))]
        axes[2].barh(CLASSES, predictions[0] * 100, color=colors)
        axes[2].set_xlabel('Probability (%)', fontsize=12)
        axes[2].set_title('Class Probabilities', fontsize=14, fontweight='bold')
        axes[2].grid(axis='x', alpha=0.3)

        plt.tight_layout()
        plt.savefig('prediction_result.png', dpi=300, bbox_inches='tight')
        plt.show()

    return result

# =====================================================
# الخطوة 12: اختبار النموذج على صورة جديدة
# =====================================================

print("\n" + "="*60)
print("📤 ارفع صورة MRI للتحليل:")
print("="*60)

uploaded_test = files.upload()

for filename in uploaded_test.keys():
    print(f"\n🔍 تحليل: {filename}")
    result = predict_brain_tumor(filename, best_model)

    print("\n" + "="*60)
    print("📊 النتيجة التفصيلية:")
    print("="*60)
    print(f"🎯 التشخيص: {result['predicted_class']}")
    print(f"📈 مستوى الثقة: {result['confidence']:.2f}%")
    print(f"⚠️  وجود ورم: {'نعم' if result['has_tumor'] else 'لا'}")
    print(f"🔴 الخطورة: {result['severity']}")

    print("\n📊 احتماليات جميع الفئات:")
    for cls, prob in result['all_probabilities'].items():
        bar = '█' * int(prob / 5)
        print(f"  {cls:15s}: {bar} {prob:5.2f}%")

# =====================================================
# الخطوة 13: حفظ النموذج بصيغ متعددة
# =====================================================

print("\n" + "="*60)
print("💾 حفظ النموذج...")
print("="*60)

# حفظ H5
best_model.save('brain_tumor_final_model.h5')
print("✅ تم الحفظ: brain_tumor_final_model.h5")

# حفظ SavedModel
best_model.save('saved_model', save_format='tf')
print("✅ تم الحفظ: saved_model/")

# تحويل إلى TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()
with open('brain_tumor_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("✅ تم الحفظ: brain_tumor_model.tflite")

# تحويل إلى TensorFlow.js
!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=keras \
    brain_tumor_final_model.h5 \
    tfjs_model/
print("✅ تم الحفظ: tfjs_model/")

# =====================================================
# الخطوة 14: تنزيل الملفات
# =====================================================

print("\n" + "="*60)
print("📥 تنزيل الملفات...")
print("="*60)

files.download('brain_tumor_final_model.h5')
files.download('brain_tumor_model.tflite')
!zip -r tfjs_model.zip tfjs_model/
files.download('tfjs_model.zip')
files.download('training_results.png')
files.download('confusion_matrix.png')

print("\n✅ تم تنزيل جميع الملفات!")

# =====================================================
# الخطوة 15: إنشاء API بسيط
# =====================================================

print("\n" + "="*60)
print("🔧 إنشاء ملف API...")
print("="*60)

api_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
import tensorflow as tf
import numpy as np
import cv2
import base64

app = Flask(__name__)
CORS(app)

# تحميل النموذج
model = tf.keras.models.load_model('brain_tumor_final_model.h5')
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = 224

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # استقبال الصورة
        data = request.json
        image_data = data['image']

        # فك تشفير base64
        image_bytes = base64.b64decode(image_data.split(',')[1])
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # تحضير الصورة
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = np.expand_dims(img / 255.0, axis=0)

        # التنبؤ
        predictions = model.predict(img)
        predicted_class = CLASSES[np.argmax(predictions[0])]
        confidence = float(np.max(predictions[0]) * 100)

        return jsonify({
            'success': True,
            'predicted_class': predicted_class,
            'confidence': confidence,
            'has_tumor': predicted_class != 'notumor',
            'all_probabilities': {CLASSES[i]: float(predictions[0][i] * 100)
                                 for i in range(len(CLASSES))}
        })

    except Exception as e:
        return jsonify({'success': False, 'error': str(e)})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
'''

with open('brain_tumor_api.py', 'w') as f:
    f.write(api_code)

files.download('brain_tumor_api.py')
print("✅ تم إنشاء: brain_tumor_api.py")

# =====================================================
# ملخص نهائي
# =====================================================

print("\n" + "="*60)
print("🎉 تم الانتهاء بنجاح!")
print("="*60)
print("\n📦 الملفات المحفوظة:")
print("  1. brain_tumor_final_model.h5 - النموذج الكامل")
print("  2. brain_tumor_model.tflite - للهواتف")
print("  3. tfjs_model.zip - للمتصفح")
print("  4. brain_tumor_api.py - API للنشر")
print("  5. training_results.png - نتائج التدريب")
print("  6. confusion_matrix.png - مصفوفة الارتباك")

print("\n🚀 الخطوات التالية:")
print("  1. استخدم brain_tumor_final_model.h5 في Python")
print("  2. استخدم tfjs_model/ في موقع الويب")
print("  3. نشر brain_tumor_api.py على سيرفر")
print("\n💡 دقة النموذج: ~94-97%")
print("="*60)